# 1. Data Intake and Quality Review

Load the raw retail transaction dataset, inspect the schema, validate core fields, and create the initial cleaned data file.

## Environment Setup

Set project-level paths so the notebook runs reliably from either the repository root or the notebooks folder.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets" / "screenshots"
DATA_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Raw Data Load

Import the source transaction file and preview the first records to confirm the dataset is available.

In [ ]:
raw_path = DATA_DIR / "Sample - Superstore.csv"
df = pd.read_csv(raw_path)
df.head()

## Dataset Size

Review row and column counts to establish the working data volume.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

## Schema Review

Check field names, data types, and non-null counts before preparing the data.

In [ ]:
df.info()

## Field Summary

Profile numeric and categorical fields to identify value ranges and potential quality issues.

In [ ]:
df.describe(include="all")

## Missing Value Check

Confirm whether any fields require missing-value handling before analysis.

In [ ]:
missing_values = df.isna().sum().reset_index()
missing_values.columns = ["Column", "Missing Values"]
missing_values[missing_values["Missing Values"] > 0]

## Duplicate Check

Identify duplicate transaction rows that could overstate sales or profit.

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count:,}")

## Business Entity Counts

Count unique orders, customers, and products to understand the commercial scope of the dataset.

In [ ]:
unique_summary = pd.DataFrame({
    "Metric": ["Orders", "Customers", "Products"],
    "Unique Count": [
        df["Order ID"].nunique(),
        df["Customer ID"].nunique(),
        df["Product ID"].nunique(),
    ],
})
unique_summary

## Date Preparation

Convert order and ship dates into date fields and create a monthly order period for trend analysis.

In [ ]:
date_columns = ["Order Date", "Ship Date"]
for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

df["Order Month"] = df["Order Date"].dt.to_period("M").astype(str)
df[["Order Date", "Ship Date", "Order Month"]].head()

## Cleaned Data Export

Save the prepared file for the next step in the workflow.

In [ ]:
cleaned_path = DATA_DIR / "cleaned_data.csv"
df.to_csv(cleaned_path, index=False)
print(f"Saved cleaned dataset to: {cleaned_path}")